# Tehran Network Project

**Authors:** Zahra Gharib & Seyed Sajjad Qavami

This collaborative Advanced Programming project explores Tehran's road network using OpenStreetMap data.

## Part 1 — Interactive Shortest Path

This section implements an object-oriented interactive route finder. The user selects two points on a Tehran map, the points are matched to nearby road-network nodes, and the shortest route by road length is drawn on the map.

In [ ]:
from pyrosm import OSM, get_data
import osmnx as ox
from ipyleaflet import Map, Marker, Polyline

class ShortestPath:
    def __init__(self):
        self.current_route = None
        self.markers = []
        osm = OSM(get_data("Tehran"))
        nodes, edges = osm.get_network(nodes=True, network_type="driving")
        self.graph = osm.to_graph(nodes, edges, graph_type="networkx")
        center = (35.6892, 51.3890)
        self.my_map = Map(center=center, zoom=15)

        self.my_map.on_interaction(self.click_handler)


    def click_handler(self, **kwargs):
        if kwargs.get("type") == "click":
            lat, lon = kwargs.get("coordinates")
            
            self.add_marker(lat, lon)

            if len(self.markers) > 2:
                self.clear_markers()
                return
            
            if len(self.markers) == 2:
                self.draw_marker_on_graph()
            

    def clear_markers(self):
        for marker in self.markers:
            self.my_map.remove_layer(marker) 
        self.markers.clear()

        if self.current_route:
            self.my_map.remove_layer(self.current_route)
            self.current_route = None
        return


    def add_marker(self, lat, lon):
        marker = Marker(location= (lat, lon), draggable=False)
        self.markers.append(marker)
        self.my_map.add_layer(marker)


    def draw_marker_on_graph(self):
        self.origin = self.markers[0].location #(lat, lon)
        self.dest = self.markers[1].location   #(lat, lon)

        self.origin_node = ox.distance.nearest_nodes(self.graph, self.origin[1], self.origin[0])
        self.dest_node = ox.distance.nearest_nodes(self.graph, self.dest[1], self.dest[0])

        self.route = ox.shortest_path(self.graph, self.origin_node, self.dest_node, weight="length")


        route_coord = []
        for node_id in self.route:
            node = self.graph.nodes[node_id]
            route_coord.append((node['y'], node['x']))
        
        if self.current_route:
            self.my_map.remove_layer(self.current_route)
        self.current_route = Polyline(locations= route_coord, color="red", weight=7)
        self.my_map.add_layer(self.current_route)

my_shortest_path = ShortestPath()
display(my_shortest_path.my_map)

### How Part 1 Works

1. **Load the network and map**
   - `pyrosm.get_data("Tehran")` provides the Tehran OpenStreetMap dataset.
   - The driving network is converted to a NetworkX graph.
   - `ipyleaflet.Map` displays the interactive map.

2. **Handle map clicks**
   - `click_handler` receives the coordinates of each clicked point.
   - The first two clicks are stored as markers.
   - A third click clears the previous markers and route so a new pair can be selected.

3. **Add and clear markers**
   - `add_marker` creates and displays a marker.
   - `clear_markers` removes existing markers and the current route.

4. **Find and draw the shortest route**
   - The selected origin and destination are matched to their nearest graph nodes with `ox.distance.nearest_nodes`.
   - `ox.shortest_path(..., weight="length")` finds the shortest route by road length.
   - The route node IDs are converted to latitude/longitude coordinates and displayed as an `ipyleaflet.Polyline`.

## Part 2 — University Accessibility Analysis

This section visualizes how road-network distance to the nearest university varies across Tehran.

### Steps 1–3 — Load Tehran's Driving Network

The Tehran OpenStreetMap dataset is loaded with `pyrosm`. The driving network is extracted as nodes and edges, then converted to a NetworkX graph.

In [ ]:
from pyrosm import OSM
from pyrosm import get_data
import numpy as np
import matplotlib.pyplot as plt

data = get_data("Tehran")
osm = OSM(data)
nodes, edges = osm.get_network(nodes=True, network_type="driving")
graph = osm.to_graph(nodes, edges, graph_type = "networkx")

### Step 4 — Distance to the Nearest University

University points of interest are extracted from the OpenStreetMap data. Each university is matched to its nearest graph node, then `networkx.multi_source_dijkstra_path_length` calculates the shortest road-network distance from all reachable nodes to the nearest university.

In [ ]:
import osmnx as ox
import networkx as nx

university = osm.get_pois(custom_filter={"amenity": ["university"]}).dropna(subset= ["lat", "lon"])
university= ox.nearest_nodes(graph, university["lon"],university["lat"])
path_length = nx.multi_source_dijkstra_path_length(graph, sources=set(university), weight= "length")

### Step 5 — Visualize the University Catchment

Node coordinates and their distances to the nearest university are collected and plotted with Matplotlib. Distances are converted to kilometers, and the color scale is capped at 10 km for clearer visualization.

In [ ]:

node_coords = {node: (data['x'], data['y']) for node, data in graph.nodes(data=True)}

x = []
y = []
distances = []

for node, (x_coord, y_coord) in node_coords.items():
    x.append(x_coord)
    y.append(y_coord)
    d = path_length.get(node, np.nan)
    distances.append(d)

x = np.array(x)
y = np.array(y)
distances = np.array(distances) / 1000

plt.figure(figsize=(13, 10))
sc = plt.scatter(x, y,c = distances, cmap="RdYlBu", s=1, alpha=0.8, vmax=10)
cbar = plt.colorbar(sc)
cbar.set_label("Distance to Nearest University (km)")
plt.title("University Catchment in Tehran")
plt.axis("off")
plt.show()
